In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/spaceship-titanic/sample_submission.csv
/kaggle/input/spaceship-titanic/train.csv
/kaggle/input/spaceship-titanic/test.csv


In [2]:
train_data = pd.read_csv('../input/spaceship-titanic/train.csv')
test_data = pd.read_csv('../input/spaceship-titanic/test.csv')

In [3]:
train_data.isnull().sum()
train_data = train_data.drop(['Cabin','Name'],axis=1)
test_data = test_data.drop(['Cabin','Name'],axis=1)

In [4]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan,strategy='most_frequent')
train_data = pd.DataFrame(imputer.fit_transform(train_data),columns=train_data.columns)
test_data = pd.DataFrame(imputer.fit_transform(test_data),columns=test_data.columns)


In [5]:
train_data['CryoSleep'] = train_data['CryoSleep'].astype('int')
test_data['CryoSleep'] = test_data['CryoSleep'].astype('int')
train_data['VIP'] = train_data['VIP'].astype('int')
test_data['VIP'] = test_data['VIP'].astype('int')
train_data['Transported'] = train_data['Transported'].astype('int')


In [6]:
train_data.head()

,PassengerId,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,0001_01,Europa,0,TRAPPIST-1e,39.0,0,0.0,0.0,0.0,0.0,0.0,0
1,0002_01,Earth,0,TRAPPIST-1e,24.0,0,109.0,9.0,25.0,549.0,44.0,1
2,0003_01,Europa,0,TRAPPIST-1e,58.0,1,43.0,3576.0,0.0,6715.0,49.0,0
3,0003_02,Europa,0,TRAPPIST-1e,33.0,0,0.0,1283.0,371.0,3329.0,193.0,0
4,0004_01,Earth,0,TRAPPIST-1e,16.0,0,303.0,70.0,151.0,565.0,2.0,1


In [7]:
train_data = pd.get_dummies(train_data,columns=['HomePlanet','Destination'])
test_data = pd.get_dummies(test_data,columns=['HomePlanet','Destination'])

In [8]:
train_data.isnull().sum()

PassengerId                  0
CryoSleep                    0
Age                          0
VIP                          0
RoomService                  0
FoodCourt                    0
ShoppingMall                 0
Spa                          0
VRDeck                       0
Transported                  0
HomePlanet_Earth             0
HomePlanet_Europa            0
HomePlanet_Mars              0
Destination_55 Cancri e      0
Destination_PSO J318.5-22    0
Destination_TRAPPIST-1e      0
dtype: int64

In [9]:
X_train = train_data.drop(['Transported'],axis=1)
y_train = train_data['Transported']
X_test = test_data

In [10]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
rf.fit(X_train,y_train)


RandomForestClassifier()

In [11]:
y_pred = rf.predict(X_test)
y_pred = pd.Series(y_pred, name="Transported" )
y_pred

0       1
1       0
2       1
3       1
4       1
       ..
4272    0
4273    0
4274    1
4275    1
4276    0
Name: Transported, Length: 4277, dtype: int64

In [12]:
y_pass = pd.Series(test_data['PassengerId'])
y_pass

0       0013_01
1       0018_01
2       0019_01
3       0021_01
4       0023_01
         ...   
4272    9266_02
4273    9269_01
4274    9271_01
4275    9273_01
4276    9277_01
Name: PassengerId, Length: 4277, dtype: object

In [13]:
y_pred = pd.concat([y_pass,y_pred],axis=1)
y_pred.head()

,PassengerId,Transported
0,0013_01,1
1,0018_01,0
2,0019_01,1
3,0021_01,1
4,0023_01,1


In [14]:
y_pred["Transported"] = y_pred["Transported"].astype('bool')
y_pred.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


In [15]:
y_pred.to_csv('submission.csv',index=False)